# MLP Hyperparameter Tuning

This notebook replaces the single fixed MLP baseline run with a **bounded, CNN/LSTM-style tuning protocol**.

Main fairness rule:
- fixed MLP architecture
- same data split and trainer
- same small candidate grid style as CNN/LSTM
- model selection on clean validation score only
- optional 3-seed confirmation after choosing the best candidate

This is intentionally **not** architecture search. The goal is a reasonably tuned MLP under the same discipline as CNN/LSTM.

Import policy: this notebook imports the original project modules under `src/models/` and does not depend on any alternate file suffixes or renamed class aliases.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing 'src'. Run this notebook from inside the project repo.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) in sys.path:
    sys.path.remove(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project


In [2]:
import random
import numpy as np
import torch

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Keep deterministic behavior when possible.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [3]:
import pandas as pd
import torch

from src.data_prep import prepare_uji_data
from src.models.mlp_coordinates import CoordinateMLPModel
from src.models.mlp_joint import JointMLPModel
from src.models.mlp_multitask import MultiTaskMLPModel
from src.training import TrainConfig, train_from_tensors

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [4]:
bundle = prepare_uji_data()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

joint_y_train, joint_y_val = bundle.get_targets(["joint"])
mt_y_train, mt_y_val = bundle.get_targets(["building", "floor"])
coord_y_train, coord_y_val = bundle.get_targets(["longitude", "latitude"])

in_dim = bundle.X_train.shape[1]

print("device:", device)
print("X train/val:", bundle.X_train.shape, bundle.X_val.shape)
print("coordinate_std:", bundle.coordinate_std)


device: cuda
X train/val: (19937, 1040) (1111, 1040)
coordinate_std: [123.39891  66.94215]


In [5]:
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Parameter-count sanity check


In [6]:
param_rows = [
    {"model": "mlp_joint", "params": count_trainable_params(JointMLPModel(in_dim=in_dim))},
    {"model": "mlp_multitask", "params": count_trainable_params(MultiTaskMLPModel(in_dim=in_dim))},
    {"model": "mlp_coordinate", "params": count_trainable_params(CoordinateMLPModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std))},
]

param_df = pd.DataFrame(param_rows)
param_df["params_millions"] = param_df["params"] / 1_000_000
param_df


,model,params,params_millions
0,mlp_joint,1729037,1.729037
1,mlp_multitask,1727752,1.727752
2,mlp_coordinate,1726210,1.726210


## Shared helpers


In [7]:
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def make_cfg(
    *,
    lr=2e-3,
    weight_decay=1e-4,
    batch_size=256,
    val_batch_size=512,
    max_epochs=30,
    patience=5,
    print_every=5,
    grad_clip_norm=None,
    run_name=None,
):
    return TrainConfig(
        lr=lr,
        weight_decay=weight_decay,
        batch_size=batch_size,
        val_batch_size=val_batch_size,
        max_epochs=max_epochs,
        patience=patience,
        print_every=print_every,
        grad_clip_norm=grad_clip_norm,
        run_name=run_name,
    )

def result_row(name, result):
    row = {"model": name, "best_epoch": result.best_epoch}
    row.update(result.best_metrics)
    return row

def run_trial(model, y_train, y_val, cfg, *, family, task, seed=42):
    set_seed(seed)
    result = train_from_tensors(
        model=model,
        X_train=bundle.X_train,
        y_train=y_train,
        X_val=bundle.X_val,
        y_val=y_val,
        device=device,
        cfg=cfg,
    )
    row = {
        "family": family,
        "task": task,
        "run_name": cfg.run_name,
        "seed": seed,
        "lr": cfg.lr,
        "weight_decay": cfg.weight_decay,
        "batch_size": cfg.batch_size,
        "val_batch_size": cfg.val_batch_size,
        "max_epochs": cfg.max_epochs,
        "patience": cfg.patience,
        "print_every": cfg.print_every,
        "grad_clip_norm": cfg.grad_clip_norm,
        "best_epoch": result.best_epoch,
    }
    row.update(result.best_metrics)
    return row

def task_data(task_name):
    if task_name == "joint":
        return joint_y_train, joint_y_val
    if task_name == "multitask":
        return mt_y_train, mt_y_val
    if task_name == "coordinate":
        return coord_y_train, coord_y_val
    raise ValueError(task_name)

def build_model(family, task_name):
    if family != "mlp":
        raise ValueError(family)
    if task_name == "joint":
        return JointMLPModel(in_dim=in_dim)
    if task_name == "multitask":
        return MultiTaskMLPModel(in_dim=in_dim)
    if task_name == "coordinate":
        return CoordinateMLPModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std)
    raise ValueError(task_name)


## Tuning grid

This grid is the final bounded-tuning protocol for the main fair comparison.

Rules:
- fixed MLP architecture
- no width/depth/dropout architecture search
- five optimizer/schedule candidates per task
- selection on clean validation score only
- 3-seed confirmation after selection

This is deliberately close to the CNN tuning discipline.

In [8]:
BASELINE_CFG = dict(
    lr=2e-3,
    weight_decay=1e-4,
    batch_size=256,
    val_batch_size=512,
    max_epochs=30,
    patience=5,
    print_every=5,
    grad_clip_norm=None,
)

# Bounded 5-candidate grid.
# This is not architecture search. The MLP architecture stays fixed.
# We only vary optimizer/schedule settings.
def mlp_specs(prefix: str):
    return [
        dict(run_name=f"{prefix}_base_2e3_wd1e4",        lr=2e-3, weight_decay=1e-4, max_epochs=30, patience=5,  grad_clip_norm=None),
        dict(run_name=f"{prefix}_stable_1e3_wd1e4",     lr=1e-3, weight_decay=1e-4, max_epochs=50, patience=10, grad_clip_norm=None),
        dict(run_name=f"{prefix}_reg_1e3_wd5e4",        lr=1e-3, weight_decay=5e-4, max_epochs=50, patience=10, grad_clip_norm=None),
        dict(run_name=f"{prefix}_low_5e4_wd1e4",        lr=5e-4, weight_decay=1e-4, max_epochs=60, patience=12, grad_clip_norm=None),
        dict(run_name=f"{prefix}_lowreg_5e4_wd5e4",     lr=5e-4, weight_decay=5e-4, max_epochs=80, patience=15, grad_clip_norm=None),
    ]

MLP_GRIDS = {
    "joint": mlp_specs("mlp_joint"),
    "multitask": mlp_specs("mlp_mt"),
    "coordinate": mlp_specs("mlp_coord"),
}

BASELINE_CFG, MLP_GRIDS

({'lr': 0.002,
  'weight_decay': 0.0001,
  'batch_size': 256,
  'val_batch_size': 512,
  'max_epochs': 30,
  'patience': 5,
  'print_every': 5,
  'grad_clip_norm': None},
 {'joint': [{'run_name': 'mlp_joint_base_2e3_wd1e4',
    'lr': 0.002,
    'weight_decay': 0.0001,
    'max_epochs': 30,
    'patience': 5,
    'grad_clip_norm': None},
   {'run_name': 'mlp_joint_stable_1e3_wd1e4',
    'lr': 0.001,
    'weight_decay': 0.0001,
    'max_epochs': 50,
    'patience': 10,
    'grad_clip_norm': None},
   {'run_name': 'mlp_joint_reg_1e3_wd5e4',
    'lr': 0.001,
    'weight_decay': 0.0005,
    'max_epochs': 50,
    'patience': 10,
    'grad_clip_norm': None},
   {'run_name': 'mlp_joint_low_5e4_wd1e4',
    'lr': 0.0005,
    'weight_decay': 0.0001,
    'max_epochs': 60,
    'patience': 12,
    'grad_clip_norm': None},
   {'run_name': 'mlp_joint_lowreg_5e4_wd5e4',
    'lr': 0.0005,
    'weight_decay': 0.0005,
    'max_epochs': 80,
    'patience': 15,
    'grad_clip_norm': None}],
  'multitask': [

## Run MLP tuning


In [9]:
RUN_GRID = True
GRID_SEED = 42

mlp_all_results = []

if RUN_GRID:
    for task_name, trials in MLP_GRIDS.items():
        y_train, y_val = task_data(task_name)
        print(f"\n===== MLP tuning: {task_name} =====")
        for spec in trials:
            print(f"\n--- {spec['run_name']} ---")
            set_seed(GRID_SEED)
            model = build_model("mlp", task_name)
            cfg = make_cfg(**spec)
            row = run_trial(model, y_train, y_val, cfg, family="mlp", task=task_name, seed=GRID_SEED)
            row["params"] = count_trainable_params(model)
            mlp_all_results.append(row)

mlp_tuning_df = pd.DataFrame(mlp_all_results)
mlp_tuning_df



===== MLP tuning: joint =====

--- mlp_joint_base_2e3_wd1e4 ---
epoch=001 train_loss=0.2610 val_loss=0.4217 score=0.8821
epoch=005 train_loss=0.0290 val_loss=0.5554 score=0.8848
epoch=010 train_loss=0.0148 val_loss=0.5304 score=0.8884

--- mlp_joint_stable_1e3_wd1e4 ---
epoch=001 train_loss=0.3307 val_loss=0.3942 score=0.8722
epoch=005 train_loss=0.0321 val_loss=0.4744 score=0.8830
epoch=010 train_loss=0.0150 val_loss=0.6331 score=0.8812
epoch=015 train_loss=0.0075 val_loss=0.5840 score=0.8902

--- mlp_joint_reg_1e3_wd5e4 ---
epoch=001 train_loss=0.3307 val_loss=0.3945 score=0.8722
epoch=005 train_loss=0.0320 val_loss=0.4201 score=0.9001
epoch=010 train_loss=0.0161 val_loss=0.5098 score=0.8929
epoch=015 train_loss=0.0073 val_loss=0.5157 score=0.8992

--- mlp_joint_low_5e4_wd1e4 ---
epoch=001 train_loss=0.4803 val_loss=0.4042 score=0.8686
epoch=005 train_loss=0.0279 val_loss=0.5088 score=0.8722
epoch=010 train_loss=0.0129 val_loss=0.4828 score=0.8956
epoch=015 train_loss=0.0127 val_los

,family,task,run_name,seed,lr,weight_decay,batch_size,val_batch_size,max_epochs,patience,print_every,grad_clip_norm,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy,params,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,mlp,joint,mlp_joint_base_2e3_wd1e4,42,0.0020,0.0001,256,512,30,5,5,None,6,6.0,0.023100,0.512539,0.898290,0.898290,0.998200,0.898290,1729037,NaN,NaN,NaN,NaN
1,mlp,joint,mlp_joint_stable_1e3_wd1e4,42,0.0010,0.0001,256,512,50,10,5,None,7,7.0,0.020839,0.501686,0.897390,0.897390,0.996400,0.897390,1729037,NaN,NaN,NaN,NaN
2,mlp,joint,mlp_joint_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,None,6,6.0,0.019622,0.419937,0.907291,0.907291,0.998200,0.907291,1729037,NaN,NaN,NaN,NaN
3,mlp,joint,mlp_joint_low_5e4_wd1e4,42,0.0005,0.0001,256,512,60,12,5,None,13,13.0,0.012638,0.492910,0.902790,0.902790,0.994599,0.902790,1729037,NaN,NaN,NaN,NaN
4,mlp,joint,mlp_joint_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,None,42,42.0,0.004395,0.536195,0.904590,0.904590,0.995500,0.904590,1729037,NaN,NaN,NaN,NaN
5,mlp,multitask,mlp_mt_base_2e3_wd1e4,42,0.0020,0.0001,256,512,30,5,5,None,5,5.0,0.030566,0.550668,0.891989,0.891989,0.999100,0.891989,1727752,NaN,NaN,NaN,NaN
6,mlp,multitask,mlp_mt_stable_1e3_wd1e4,42,0.0010,0.0001,256,512,50,10,5,None,8,8.0,0.023392,0.493092,0.898290,0.898290,0.999100,0.898290,1727752,NaN,NaN,NaN,NaN
7,mlp,multitask,mlp_mt_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,None,20,20.0,0.008897,0.564936,0.909091,0.909091,0.999100,0.909091,1727752,NaN,NaN,NaN,NaN
8,mlp,multitask,mlp_mt_low_5e4_wd1e4,42,0.0005,0.0001,256,512,60,12,5,None,6,6.0,0.026185,0.423287,0.900990,0.900990,0.999100,0.900990,1727752,NaN,NaN,NaN,NaN
9,mlp,multitask,mlp_mt_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,None,8,8.0,0.020647,0.435436,0.907291,0.907291,0.999100,0.907291,1727752,NaN,NaN,NaN,NaN


## Select best candidate per task


In [10]:
summary_cols = [
    "family", "task", "run_name", "seed", "best_epoch", "score",
    "joint_accuracy", "building_accuracy", "floor_accuracy",
    "coordinate_mean_euclidean_m", "coordinate_rmse_m",
    "lr", "weight_decay", "max_epochs", "patience", "grad_clip_norm", "params"
]

display(mlp_tuning_df[[c for c in summary_cols if c in mlp_tuning_df.columns]].sort_values(["task", "score"], ascending=[True, False]))

best_by_family_task = (
    mlp_tuning_df
    .sort_values(["family", "task", "score"], ascending=[True, True, False])
    .groupby(["family", "task"], as_index=False)
    .first()
)

best_by_family_task


,family,task,run_name,seed,best_epoch,score,joint_accuracy,building_accuracy,floor_accuracy,coordinate_mean_euclidean_m,coordinate_rmse_m,lr,weight_decay,max_epochs,patience,grad_clip_norm,params
14,mlp,coordinate,mlp_coord_lowreg_5e4_wd5e4,42,78,-9.793972,NaN,NaN,NaN,9.793972,13.038949,0.0005,0.0005,80,15,None,1726210
11,mlp,coordinate,mlp_coord_stable_1e3_wd1e4,42,49,-9.883503,NaN,NaN,NaN,9.883503,13.036923,0.0010,0.0001,50,10,None,1726210
12,mlp,coordinate,mlp_coord_reg_1e3_wd5e4,42,47,-9.913641,NaN,NaN,NaN,9.913641,13.166144,0.0010,0.0005,50,10,None,1726210
13,mlp,coordinate,mlp_coord_low_5e4_wd1e4,42,60,-10.105044,NaN,NaN,NaN,10.105044,13.271347,0.0005,0.0001,60,12,None,1726210
10,mlp,coordinate,mlp_coord_base_2e3_wd1e4,42,10,-11.312214,NaN,NaN,NaN,11.312214,14.847737,0.0020,0.0001,30,5,None,1726210
2,mlp,joint,mlp_joint_reg_1e3_wd5e4,42,6,0.907291,0.907291,0.998200,0.907291,NaN,NaN,0.0010,0.0005,50,10,None,1729037
4,mlp,joint,mlp_joint_lowreg_5e4_wd5e4,42,42,0.904590,0.904590,0.995500,0.904590,NaN,NaN,0.0005,0.0005,80,15,None,1729037
3,mlp,joint,mlp_joint_low_5e4_wd1e4,42,13,0.902790,0.902790,0.994599,0.902790,NaN,NaN,0.0005,0.0001,60,12,None,1729037
0,mlp,joint,mlp_joint_base_2e3_wd1e4,42,6,0.898290,0.898290,0.998200,0.898290,NaN,NaN,0.0020,0.0001,30,5,None,1729037
1,mlp,joint,mlp_joint_stable_1e3_wd1e4,42,7,0.897390,0.897390,0.996400,0.897390,NaN,NaN,0.0010,0.0001,50,10,None,1729037


,family,task,run_name,seed,lr,weight_decay,batch_size,val_batch_size,max_epochs,patience,print_every,grad_clip_norm,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy,params,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,mlp,coordinate,mlp_coord_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,None,78,78.0,0.009859,0.010967,-9.793972,NaN,NaN,NaN,1726210,0.110241,0.146278,9.793972,13.038949
1,mlp,joint,mlp_joint_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,None,6,6.0,0.019622,0.419937,0.907291,0.907291,0.9982,0.907291,1729037,NaN,NaN,NaN,NaN
2,mlp,multitask,mlp_mt_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,None,20,20.0,0.008897,0.564936,0.909091,0.909091,0.9991,0.909091,1727752,NaN,NaN,NaN,NaN


## 3-seed confirmation for selected MLP configs


In [11]:
RUN_CONFIRM_FINAL = True
SEEDS_CONFIRM = (42, 123, 999)

confirm_rows = []

if RUN_CONFIRM_FINAL:
    for _, best in best_by_family_task.iterrows():
        task_name = best["task"]
        y_train, y_val = task_data(task_name)

        spec = {
            "run_name": f"confirm_{best['run_name']}",
            "lr": float(best["lr"]),
            "weight_decay": float(best["weight_decay"]),
            "max_epochs": int(best["max_epochs"]),
            "patience": int(best["patience"]),
            "print_every": 5,
            "grad_clip_norm": None if pd.isna(best.get("grad_clip_norm", None)) else float(best["grad_clip_norm"]),
        }

        for seed in SEEDS_CONFIRM:
            print(f"\n===== Confirm MLP {task_name}, seed={seed} =====")
            set_seed(seed)
            model = build_model("mlp", task_name)
            cfg = make_cfg(**spec)
            row = run_trial(model, y_train, y_val, cfg, family="mlp", task=task_name, seed=seed)
            row["selected_from_run"] = best["run_name"]
            row["params"] = count_trainable_params(model)
            confirm_rows.append(row)

mlp_confirm_runs_df = pd.DataFrame(confirm_rows)
mlp_confirm_runs_df



===== Confirm MLP coordinate, seed=42 =====
epoch=001 train_loss=0.1232 val_loss=0.0275 score=-18.2084
epoch=005 train_loss=0.0276 val_loss=0.0188 score=-14.2839
epoch=010 train_loss=0.0199 val_loss=0.0193 score=-15.2477
epoch=015 train_loss=0.0159 val_loss=0.0138 score=-11.3452
epoch=020 train_loss=0.0146 val_loss=0.0138 score=-11.3100
epoch=025 train_loss=0.0141 val_loss=0.0126 score=-10.5721
epoch=030 train_loss=0.0143 val_loss=0.0116 score=-10.3791
epoch=035 train_loss=0.0125 val_loss=0.0120 score=-10.4514
epoch=040 train_loss=0.0122 val_loss=0.0124 score=-10.9530
epoch=045 train_loss=0.0110 val_loss=0.0114 score=-10.0062
epoch=050 train_loss=0.0107 val_loss=0.0114 score=-10.0593
epoch=055 train_loss=0.0106 val_loss=0.0115 score=-10.0057
epoch=060 train_loss=0.0102 val_loss=0.0114 score=-9.9488
epoch=065 train_loss=0.0103 val_loss=0.0113 score=-9.8645
epoch=070 train_loss=0.0100 val_loss=0.0112 score=-9.9011
epoch=075 train_loss=0.0101 val_loss=0.0112 score=-9.8908
epoch=080 train

,family,task,run_name,seed,lr,weight_decay,batch_size,val_batch_size,max_epochs,patience,print_every,grad_clip_norm,best_epoch,epoch,train_loss,val_loss,score,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m,selected_from_run,params,joint_accuracy,building_accuracy,floor_accuracy
0,mlp,coordinate,confirm_mlp_coord_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,None,78,78.0,0.009859,0.010967,-9.793972,0.110241,0.146278,9.793972,13.038949,mlp_coord_lowreg_5e4_wd5e4,1726210,NaN,NaN,NaN
1,mlp,coordinate,confirm_mlp_coord_lowreg_5e4_wd5e4,123,0.0005,0.0005,256,512,80,15,5,None,51,51.0,0.011222,0.011679,-10.133181,0.114208,0.151283,10.133181,13.344146,mlp_coord_lowreg_5e4_wd5e4,1726210,NaN,NaN,NaN
2,mlp,coordinate,confirm_mlp_coord_lowreg_5e4_wd5e4,999,0.0005,0.0005,256,512,80,15,5,None,63,63.0,0.010904,0.011986,-10.277055,0.115508,0.153224,10.277055,13.473429,mlp_coord_lowreg_5e4_wd5e4,1726210,NaN,NaN,NaN
3,mlp,joint,confirm_mlp_joint_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,None,6,6.0,0.019622,0.419937,0.907291,NaN,NaN,NaN,NaN,mlp_joint_reg_1e3_wd5e4,1729037,0.907291,0.9982,0.907291
4,mlp,joint,confirm_mlp_joint_reg_1e3_wd5e4,123,0.0010,0.0005,256,512,50,10,5,None,17,17.0,0.008643,0.551749,0.906391,NaN,NaN,NaN,NaN,mlp_joint_reg_1e3_wd5e4,1729037,0.906391,0.9973,0.906391
5,mlp,joint,confirm_mlp_joint_reg_1e3_wd5e4,999,0.0010,0.0005,256,512,50,10,5,None,10,10.0,0.017675,0.507573,0.904590,NaN,NaN,NaN,NaN,mlp_joint_reg_1e3_wd5e4,1729037,0.904590,0.9964,0.904590
6,mlp,multitask,confirm_mlp_mt_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,None,20,20.0,0.008897,0.564936,0.909091,NaN,NaN,NaN,NaN,mlp_mt_reg_1e3_wd5e4,1727752,0.909091,0.9991,0.909091
7,mlp,multitask,confirm_mlp_mt_reg_1e3_wd5e4,123,0.0010,0.0005,256,512,50,10,5,None,16,16.0,0.009232,0.507648,0.909091,NaN,NaN,NaN,NaN,mlp_mt_reg_1e3_wd5e4,1727752,0.909091,0.9991,0.909091
8,mlp,multitask,confirm_mlp_mt_reg_1e3_wd5e4,999,0.0010,0.0005,256,512,50,10,5,None,19,19.0,0.009274,0.555492,0.906391,NaN,NaN,NaN,NaN,mlp_mt_reg_1e3_wd5e4,1727752,0.906391,0.9982,0.906391


## Confirmation summary


In [12]:
def summarize_confirm(df):
    if df.empty:
        return pd.DataFrame()
    agg = {
        "seed": "count",
        "score": ["mean", "std"],
        "best_epoch": "mean",
        "train_loss": "mean",
        "val_loss": "mean",
        "joint_accuracy": "mean",
        "building_accuracy": "mean",
        "floor_accuracy": "mean",
        "coordinate_mean_euclidean_m": "mean",
        "coordinate_rmse_m": "mean",
        "params": "first",
    }
    available_agg = {k: v for k, v in agg.items() if k in df.columns}
    out = df.groupby(["family", "task", "run_name"]).agg(available_agg)
    out.columns = ["_".join(col).strip("_") if isinstance(col, tuple) else col for col in out.columns]
    out = out.reset_index().rename(columns={"seed_count": "runs"})
    return out

mlp_confirmation_summary = summarize_confirm(mlp_confirm_runs_df)
mlp_confirmation_summary


,family,task,run_name,runs,score_mean,score_std,best_epoch_mean,train_loss_mean,val_loss_mean,joint_accuracy_mean,building_accuracy_mean,floor_accuracy_mean,coordinate_mean_euclidean_m_mean,coordinate_rmse_m_mean,params_first
0,mlp,coordinate,confirm_mlp_coord_lowreg_5e4_wd5e4,3,-10.068069,0.248036,64.000000,0.010662,0.011544,NaN,NaN,NaN,10.068069,13.285508,1726210
1,mlp,joint,confirm_mlp_joint_reg_1e3_wd5e4,3,0.906091,0.001375,11.000000,0.015313,0.493086,0.906091,0.9973,0.906091,NaN,NaN,1729037
2,mlp,multitask,confirm_mlp_mt_reg_1e3_wd5e4,3,0.908191,0.001559,18.333333,0.009134,0.542692,0.908191,0.9988,0.908191,NaN,NaN,1727752


## Final config dictionary to copy into baseline/robustness notebooks


In [13]:
FINAL_TUNED_CFGS = {
    family: {
        row["task"]: {
            "lr": float(row["lr"]),
            "weight_decay": float(row["weight_decay"]),
            "grad_clip_norm": None if pd.isna(row.get("grad_clip_norm", None)) else float(row["grad_clip_norm"]),
            "max_epochs": int(row["max_epochs"]),
            "patience": int(row["patience"]),
            "print_every": 5,
            "batch_size": 256,
            "val_batch_size": 512,
        }
        for _, row in g.iterrows()
    }
    for family, g in best_by_family_task.groupby("family")
}

FINAL_TUNED_CFGS


{'mlp': {'coordinate': {'lr': 0.0005,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 80,
   'patience': 15,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'joint': {'lr': 0.001,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'multitask': {'lr': 0.001,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512}}}

## Save outputs


In [14]:
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "logs" / "fair_tuning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not mlp_tuning_df.empty:
    mlp_tuning_df.to_csv(OUTPUT_DIR / "mlp_tuning_grid_results.csv", index=False)
if not mlp_confirm_runs_df.empty:
    mlp_confirm_runs_df.to_csv(OUTPUT_DIR / "mlp_confirm_runs.csv", index=False)
if not mlp_confirmation_summary.empty:
    mlp_confirmation_summary.to_csv(OUTPUT_DIR / "mlp_confirmation_summary.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR)


Saved outputs to: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/notebooks/logs/fair_tuning
